In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from collections import Counter

# Download NLTK resources (run once)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

print("✅ All libraries ready!")

✅ All libraries ready!


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/shivam13juna/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/shivam13juna/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/shivam13juna/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/shivam13juna/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/shivam13juna/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [2]:
# Load the dataset
# Source: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

df = pd.read_csv('imdb_dataset.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Dataset shape: (50000, 2)
Columns: ['review', 'sentiment']


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
# Check the sentiment distribution
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [4]:
# Let's look at a few sample reviews
for i, row in df.head(3).iterrows():
    print(f"\n{'='*60}")
    print(f"Sentiment: {row['sentiment'].upper()}")
    print(f"Review: {row['review'][:200]}...")


Sentiment: POSITIVE
Review: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo...

Sentiment: POSITIVE
Review: A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece...

Sentiment: POSITIVE
Review: I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simplistic, but the dialogue is wi...


# Part 1

In [5]:
# Simple example: What is a token?
sentence = "I love NLP!"

# METHOD 1: Simplest approach - split by spaces
tokens_simple = sentence.split()
print("Split by space:", tokens_simple)
print("Number of tokens:", len(tokens_simple))

Split by space: ['I', 'love', 'NLP!']
Number of tokens: 3


In [6]:
# But wait... what about punctuation?
sentence = "Hello, world! How are you?"

tokens_simple = sentence.split()
print("Split by space:", tokens_simple)
# Problem: "Hello," and "you?" have punctuation attached!

Split by space: ['Hello,', 'world!', 'How', 'are', 'you?']


In [7]:
from nltk.tokenize import word_tokenize, TreebankWordTokenizer

# A complex sentence to tokenize
text = "I can't believe http://wow.com is FREE!!! It's amazing @user 🔥"

print("Original text:")
print(text)
print("\n" + "="*60)

Original text:
I can't believe http://wow.com is FREE!!! It's amazing @user 🔥



In [8]:
# Method 1: Simple whitespace split
tokens_whitespace = text.split()
print("\n1️⃣ Whitespace split:")
print(tokens_whitespace)
print(f"   → {len(tokens_whitespace)} tokens")


1️⃣ Whitespace split:
['I', "can't", 'believe', 'http://wow.com', 'is', 'FREE!!!', "It's", 'amazing', '@user', '🔥']
   → 10 tokens


In [9]:
# Method 2: NLTK's word_tokenize (smarter!)
tokens_nltk = word_tokenize(text)
print("\n2️⃣ NLTK word_tokenize:")
print(tokens_nltk)
print(f"   → {len(tokens_nltk)} tokens")
print("   → Notice: 'can't' → 'ca' + 'n't' (handles contractions!)")


2️⃣ NLTK word_tokenize:
['I', 'ca', "n't", 'believe', 'http', ':', '//wow.com', 'is', 'FREE', '!', '!', '!', 'It', "'s", 'amazing', '@', 'user', '🔥']
   → 18 tokens
   → Notice: 'can't' → 'ca' + 'n't' (handles contractions!)


In [10]:
sentence = "Hello, world! How are you?"
tokens_nltk = word_tokenize(sentence)
print("NLTK word_tokenize:", tokens_nltk)

NLTK word_tokenize: ['Hello', ',', 'world', '!', 'How', 'are', 'you', '?']


## LowerCase

In [11]:
# WHEN LOWERCASING HELPS
text1 = "I love Apple products. APPLE makes great devices."

tokens_original = word_tokenize(text1)
tokens_lower = word_tokenize(text1.lower())

print("Original tokens:", tokens_original)
print("Lowercased tokens:", tokens_lower)
print("\n✅ BENEFIT: 'Apple' and 'APPLE' are now the same token!")

Original tokens: ['I', 'love', 'Apple', 'products', '.', 'APPLE', 'makes', 'great', 'devices', '.']
Lowercased tokens: ['i', 'love', 'apple', 'products', '.', 'apple', 'makes', 'great', 'devices', '.']

✅ BENEFIT: 'Apple' and 'APPLE' are now the same token!


In [12]:
# WHEN LOWERCASING HURTS
text2 = "I work in IT. It is interesting. US policy affects us."

print("Original:", text2)
print("Lowercased:", text2.lower())

print("\n❌ PROBLEM:")
print("   'IT' (technology) → 'it' (pronoun) - meaning lost!")
print("   'US' (country) → 'us' (pronoun) - meaning lost!")

Original: I work in IT. It is interesting. US policy affects us.
Lowercased: i work in it. it is interesting. us policy affects us.

❌ PROBLEM:
   'IT' (technology) → 'it' (pronoun) - meaning lost!
   'US' (country) → 'us' (pronoun) - meaning lost!


In [13]:
df['review'].iloc[0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [14]:
# Let's see the impact on our IMDB dataset
sample_review = df['review'].iloc[0]

tokens_original = word_tokenize(sample_review)
tokens_lower = word_tokenize(sample_review.lower())

# Count unique tokens
print(f"Original unique tokens: {len(set(tokens_original))}")
print(f"Lowercased unique tokens: {len(set(tokens_lower))}")
print(f"\n✅ Vocabulary reduced by {len(set(tokens_original)) - len(set(tokens_lower))} tokens!")

Original unique tokens: 210
Lowercased unique tokens: 202

✅ Vocabulary reduced by 8 tokens!


In [15]:
# Special tokens: URLs, emails, mentions, numbers
text = "Check out https://example.com or email me@test.com! Price: $199.99 @username #hashtag"

print("Original:", text)
print("\nProblem: These special patterns can cause noise!")

Original: Check out https://example.com or email me@test.com! Price: $199.99 @username #hashtag

Problem: These special patterns can cause noise!


In [16]:
word_tokenize(text)

['Check',
 'out',
 'https',
 ':',
 '//example.com',
 'or',
 'email',
 'me',
 '@',
 'test.com',
 '!',
 'Price',
 ':',
 '$',
 '199.99',
 '@',
 'username',
 '#',
 'hashtag']

In [17]:
# Function to replace special tokens with placeholders
def normalize_special_tokens(text):
    """Replace URLs, emails, mentions, numbers with placeholders"""
    
    # Replace URLs
    # Pattern: r'https?://\S+|www\.\S+'
    # - https? : matches 'http' or 'https' (? makes 's' optional)
    # - :// : matches the literal '://' in URLs
    # - \S+ : matches one or more non-whitespace characters (the rest of URL)
    # - | : OR operator
    # - www\. : matches literal 'www.' (dot is escaped with \)
    # - \S+ : matches rest of URL after www.
    text = re.sub(r'https?://\S+|www\.\S+', '<URL>', text)
    
    # Replace emails
    # Pattern: r'\S+@\S+\.\S+'
    # - \S+ : matches one or more non-whitespace chars (username part)
    # - @ : matches literal '@' symbol
    # - \S+ : matches domain name (e.g., 'gmail')
    # - \. : matches literal dot (escaped)
    # - \S+ : matches domain extension (e.g., 'com')
    text = re.sub(r'\S+@\S+\.\S+', '<EMAIL>', text)
    
    # Replace @mentions
    # Pattern: r'@\w+'
    # - @ : matches literal '@' symbol
    # - \w+ : matches one or more word characters (letters, digits, underscore)
    # - Example: '@user123' → '<MENTION>'
    text = re.sub(r'@\w+', '<MENTION>', text)
    
    # Replace #hashtags
    # Pattern: r'#\w+'
    # - # : matches literal '#' symbol
    # - \w+ : matches one or more word characters after the hash
    # - Example: '#python' → '<HASHTAG>'
    text = re.sub(r'#\w+', '<HASHTAG>', text)
    
    # Replace money amounts
    # Pattern: r'\$[\d,]+\.?\d*'
    # - \$ : matches literal dollar sign (escaped)
    # - [\d,]+ : matches one or more digits or commas (e.g., '1,234')
    # - \.? : matches optional decimal point (? makes it 0 or 1 time)
    # - \d* : matches zero or more digits after decimal (* means 0 or more)
    # - Examples: '$199', '$1,234.56', '$50.5' → '<MONEY>'
    text = re.sub(r'\$[\d,]+\.?\d*', '<MONEY>', text)
    
    # Replace numbers
    # Pattern: r'\b\d+\b'
    # - \b : word boundary (ensures we match whole numbers, not parts)
    # - \d+ : matches one or more digits
    # - \b : word boundary at the end
    # - Example: '123' in 'I have 123 apples' → '<NUM>'
    # - Won't match '123' inside 'abc123def' due to word boundaries
    text = re.sub(r'\b\d+\b', '<NUM>', text)
    
    return text

# Apply to our example
normalized = normalize_special_tokens(text)
print("Original:")
print(text)
print("\nNormalized:")
print(normalized)


Original:
Check out https://example.com or email me@test.com! Price: $199.99 @username #hashtag

Normalized:
Check out <URL> or email <EMAIL> Price: <MONEY> <MENTION> <HASHTAG>


In [18]:
from nltk.corpus import stopwords

# Get English stopwords
stop_words = set(stopwords.words('english'))

print(f"Number of stopwords: {len(stop_words)}")
print(f"\nSome examples: {list(stop_words)[:20]}")

Number of stopwords: 198

Some examples: ['were', 'with', 'be', "he'll", 'which', "mustn't", 've', "shouldn't", 'having', "wouldn't", "it'd", 'whom', 'hadn', "you'd", 'some', "mightn't", 're', 'him', 'her', 'each']


In [19]:
# DEMONSTRATION: Why this matters
sentence = "I do not like this movie"

tokens = word_tokenize(sentence.lower())
print("Original tokens:", tokens)

# Remove stopwords (DANGEROUS!)
tokens_no_stop = [t for t in tokens if t not in stop_words]
print("After stopword removal:", tokens_no_stop)

print("\n🚨 MEANING REVERSED!")
print("   'I do not like this movie' → 'like movie'")
print("   Negative sentiment → looks positive!")

Original tokens: ['i', 'do', 'not', 'like', 'this', 'movie']
After stopword removal: ['like', 'movie']

🚨 MEANING REVERSED!
   'I do not like this movie' → 'like movie'
   Negative sentiment → looks positive!


In [20]:
# ⚠️ THE DANGER: Negation words are often in stopword lists!
negation_words = ['not', 'no', 'never', 'neither', 'nor', "n't", 'cannot', "don't", "won't", "didn't"]

print("Checking if negation words are in stopwords list:")
for word in negation_words:
    status = "⚠️ IN STOPWORDS!" if word in stop_words else "✅ Not in stopwords"
    print(f"  '{word}': {status}")

Checking if negation words are in stopwords list:
  'not': ⚠️ IN STOPWORDS!
  'no': ⚠️ IN STOPWORDS!
  'never': ✅ Not in stopwords
  'neither': ✅ Not in stopwords
  'nor': ⚠️ IN STOPWORDS!
  'n't': ✅ Not in stopwords
  'cannot': ✅ Not in stopwords
  'don't': ⚠️ IN STOPWORDS!
  'won't': ⚠️ IN STOPWORDS!
  'didn't': ⚠️ IN STOPWORDS!


In [21]:
# SOLUTION: Create a custom stopword list that PRESERVES negation
negation_words_to_keep = {'not', 'no', 'never', 'neither', 'nor', "n't", 'cannot', 
                          "don't", "won't", "didn't", "wasn't", "isn't", 
                          "aren't", "haven't", "hasn't", "hadn't", "couldn't",
                          "shouldn't", "wouldn't", "but"}

# Safe stopwords = original stopwords MINUS negation words
safe_stopwords = stop_words - negation_words_to_keep

# Now remove stopwords safely
tokens_safe = [t for t in tokens if t not in safe_stopwords]
print("With SAFE stopword removal:", tokens_safe)
print("\n✅ 'not' is preserved! Meaning intact.")

With SAFE stopword removal: ['not', 'like', 'movie']

✅ 'not' is preserved! Meaning intact.


In [22]:
# More examples
test_sentences = [
    "This is not good",
    "I would never recommend this",
    "There is no way this works",
    "I didn't like the ending"
]

print("Comparison: Standard vs Safe Stopword Removal\n")
for sent in test_sentences:
    tokens = word_tokenize(sent.lower())
    standard = [t for t in tokens if t not in stop_words]
    safe = [t for t in tokens if t not in safe_stopwords]
    
    print(f"Original: {sent}")
    print(f"  ❌ Standard: {standard}")
    print(f"  ✅ Safe:     {safe}")
    print()

Comparison: Standard vs Safe Stopword Removal

Original: This is not good
  ❌ Standard: ['good']
  ✅ Safe:     ['not', 'good']

Original: I would never recommend this
  ❌ Standard: ['would', 'never', 'recommend']
  ✅ Safe:     ['would', 'never', 'recommend']

Original: There is no way this works
  ❌ Standard: ['way', 'works']
  ✅ Safe:     ['no', 'way', 'works']

Original: I didn't like the ending
  ❌ Standard: ["n't", 'like', 'ending']
  ✅ Safe:     ["n't", 'like', 'ending']



In [23]:
# Punctuation can carry MEANING!
examples = [
    "This is great!!!",      # Strong positive
    "This is great.",        # Neutral positive
    "This is great...",      # Uncertain/hesitant
    "This is great?",        # Sarcastic/questioning
]

print("Same words, different punctuation = different meaning!\n")
for ex in examples:
    print(f"  {ex}")

Same words, different punctuation = different meaning!

  This is great!!!
  This is great.
  This is great...
  This is great?


In [24]:
# The classic example: Punctuation saves lives!
print("Let's eat grandma!!")
print("Let's eat, grandma!!")
print("\n🍽️ Grandma's life depends on that comma!")

Let's eat grandma!!
Let's eat, grandma!!

🍽️ Grandma's life depends on that comma!


In [25]:
# Function to handle punctuation thoughtfully
import string

def remove_punctuation(text, keep_important=True):
    """Remove punctuation, optionally keeping important ones"""
    if keep_important:
        # Keep ! and ? as they carry sentiment
        punctuation = string.punctuation.replace('!', '').replace('?', '')
    else:
        punctuation = string.punctuation
    
    return text.translate(str.maketrans('', '', punctuation))

text = "Wow!!! This is amazing??? I can't believe it..."
print(f"Original: {text}")
print(f"Remove all: {remove_punctuation(text, keep_important=False)}")
print(f"Keep !?: {remove_punctuation(text, keep_important=True)}")

Original: Wow!!! This is amazing??? I can't believe it...
Remove all: Wow This is amazing I cant believe it
Keep !?: Wow!!! This is amazing??? I cant believe it


## Stemming vs Lemmatization


### Very common inflectional endings

* **`-s`, `-es`**
  Examples: `cats → cat`, `caresses → caress`, `ponies → poni` (Porter often uses `i`)
* **`-ed`, `-ing`**
  Examples: `agreed → agree` (via `eed → ee`/`e` rules), `playing → play`, `hopping → hop`
* **`-y → -i`** (when there’s a vowel earlier)
  Example: `happy → happi`

### Common derivational endings (often more “word-formation” than grammar)

These show up in later steps and can be **removed** or **rewritten**:

* **`-ation/-ization/-izer`**
  Examples: `organization → organ`, `realization → realiz → real`, `computerizer → computerize → comput`
* **`-ational/-tional`**
  Examples: `relational → relat`, `conditional → condit`
* **`-ness`, `-ful`**
  Examples: `happiness → happi`, `usefulness → use`
* **`-al/-ical/-ic`**
  Examples: `logical → logic → log`, `political → polit`
* **`-ive/-ative`**
  Examples: `talkative → talk`, `decisive → decis`
* **`-ment`, `-ement`**
  Examples: `adjustment → adjust`, `replacement → replac`
* **`-able/-ible`**
  Examples: `readable → read`, `sensible → sens`
* **`-ity/-aliti/-iviti/-biliti`**
  Examples: `sensitivity → sensit`, `capability → capabl`
* **`-ous/-ousness`**
  Examples: `famous → fam`, `generousness → gener`
* **`-ism`**
  Example: `capitalism → capit`


In [26]:
from nltk.stem import PorterStemmer, SnowballStemmer
from nltk.stem import WordNetLemmatizer

# Initialize stemmers and lemmatizer
porter = PorterStemmer()
snowball = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()

print("Stemmer vs Lemmatizer: What's the difference?\n")

Stemmer vs Lemmatizer: What's the difference?



In [27]:
# Test words
words = ['running', 'runs', 'ran', 'studies', 'studying', 'better', 'cats', 'wolves', 'feet']

print(f"{'Word':<12} {'Porter Stem':<15} {'Snowball Stem':<15} {'Lemma':<12}")
print("-" * 55)

for word in words:
    stem_porter = porter.stem(word)
    stem_snowball = snowball.stem(word)
    lemma = lemmatizer.lemmatize(word)
    print(f"{word:<12} {stem_porter:<15} {stem_snowball:<15} {lemma:<12}")

Word         Porter Stem     Snowball Stem   Lemma       
-------------------------------------------------------
running      run             run             running     
runs         run             run             run         
ran          ran             ran             ran         
studies      studi           studi           study       
studying     studi           studi           studying    
better       better          better          better      
cats         cat             cat             cat         
wolves       wolv            wolv            wolf        
feet         feet            feet            foot        


In [28]:
import spacy

nlp = spacy.load('en_core_web_lg')

# Test spaCy lemmatizer
spacy_test_words = ['good', 'better', 'best', 'running', 'runs', 'ran', 'studies', 'studying', 'cats', 'wolves', 'feet']
print("spaCy Lemmatization:")
for word in spacy_test_words:
    doc = nlp(word)
    lemma_spacy = doc[0].lemma_
    print(f"Word: {word:10} | spaCy Lemma: {lemma_spacy:10}")

print("\n" + "="*60 + "\n")

spaCy Lemmatization:
Word: good       | spaCy Lemma: good      
Word: better     | spaCy Lemma: well      
Word: best       | spaCy Lemma: good      
Word: running    | spaCy Lemma: run       
Word: runs       | spaCy Lemma: run       
Word: ran        | spaCy Lemma: run       
Word: studies    | spaCy Lemma: study     
Word: studying   | spaCy Lemma: study     
Word: cats       | spaCy Lemma: cat       
Word: wolves     | spaCy Lemma: wolf      
Word: feet       | spaCy Lemma: foot      




In [29]:
def preprocess_text(text, 
                    lowercase=True,
                    remove_html=True,
                    remove_urls=True,
                    remove_stopwords=True,
                    preserve_negation=True,
                    stem=False,
                    lemmatize=False):
    """
    Complete text preprocessing pipeline.
    
    Parameters:
    -----------
    text : str - Input text
    lowercase : bool - Convert to lowercase
    remove_html : bool - Remove HTML tags
    remove_urls : bool - Remove URLs
    remove_stopwords : bool - Remove stopwords
    preserve_negation : bool - Keep negation words when removing stopwords
    stem : bool - Apply Porter stemming
    lemmatize : bool - Apply lemmatization
    
    Returns:
    --------
    list of tokens
    """
    
    # Step 1: Remove HTML
    if remove_html:
        text = re.sub(r'<[^>]+>', ' ', text)
    
    # Step 2: Remove URLs
    if remove_urls:
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Step 3: Lowercase
    if lowercase:
        text = text.lower()
    
    # Step 4: Tokenize
    tokens = word_tokenize(text)
    
    # Step 5: Remove stopwords (safely!)
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        if preserve_negation:
            negation = {'not', 'no', 'never', 'neither', 'nor', "n't", 
                       'cannot', "don't", "won't", "didn't", "wasn't", 
                       "isn't", "aren't", "haven't", "hasn't", "hadn't", 
                       "couldn't", "shouldn't", "wouldn't", "but"}
            stop_words = stop_words - negation
        tokens = [t for t in tokens if t not in stop_words]
    
    # Step 6: Remove non-alphabetic tokens
    tokens = [t for t in tokens if t.isalpha() or t == "n't"]
    
    # Step 7: Stem or Lemmatize
    if stem:
        stemmer = PorterStemmer()
        tokens = [stemmer.stem(t) for t in tokens]
    elif lemmatize:
        doc = nlp(' '.join(tokens))
        tokens = [token.lemma_ for token in doc]
    
    return tokens

In [30]:
# Apply to entire dataset
print("Processing entire dataset...")
df['tokens'] = df['review'].apply(lambda x: preprocess_text(x, lemmatize=False))
print("✅ Done!")

# Show result
df[['review', 'tokens', 'sentiment']].head()

Processing entire dataset...
✅ Done!


,review,tokens,sentiment
0,One of the other reviewers has mentioned that ...,"[one, reviewers, mentioned, watching, oz, epis...",positive
1,A wonderful little production. <br /><br />The...,"[wonderful, little, production, filming, techn...",positive
2,I thought this was a wonderful way to spend ti...,"[thought, wonderful, way, spend, time, hot, su...",positive
3,Basically there's a family where a little boy ...,"[basically, family, little, boy, jake, thinks,...",negative
4,"Petter Mattei's ""Love in the Time of Money"" is...","[petter, mattei, love, time, money, visually, ...",positive


# Part 2

## Bow

In [31]:
# Core idea: Count how many times each word appears
# Ignore word order completely!

sentence1 = "the cat sat on the mat"
sentence2 = "the dog sat on the cat"

# Manual BoW
def manual_bow(text):
    words = text.lower().split()
    return Counter(words)

bow1 = manual_bow(sentence1)
bow2 = manual_bow(sentence2)

print(f"Sentence 1: '{sentence1}'")
print(f"BoW: {dict(bow1)}")
print(f"\nSentence 2: '{sentence2}'")
print(f"BoW: {dict(bow2)}")

Sentence 1: 'the cat sat on the mat'
BoW: {'the': 2, 'cat': 1, 'sat': 1, 'on': 1, 'mat': 1}

Sentence 2: 'the dog sat on the cat'
BoW: {'the': 2, 'dog': 1, 'sat': 1, 'on': 1, 'cat': 1}


In [32]:
# Now let's create a vector representation
# Step 1: Build vocabulary from both sentences
all_words = list(set(bow1.keys()) | set(bow2.keys()))
all_words.sort()  # Sort for consistency
print("Vocabulary:", all_words)
print(f"Vocabulary size: {len(all_words)}")

Vocabulary: ['cat', 'dog', 'mat', 'on', 'sat', 'the']
Vocabulary size: 6


In [33]:
# Step 2: Create vectors
def to_vector(bow, vocabulary):
    return [bow.get(word, 0) for word in vocabulary]

vec1 = to_vector(bow1, all_words)
vec2 = to_vector(bow2, all_words)

print("Vector representation:")
print(f"Vocabulary: {all_words}")
# Visual representation
bow_df = pd.DataFrame(
    [vec1, vec2],
    columns=all_words,
    index=['"the cat sat on the mat"', '"the dog sat on the cat"']
)
print("Document-Term Matrix:")
bow_df

Vector representation:
Vocabulary: ['cat', 'dog', 'mat', 'on', 'sat', 'the']
Document-Term Matrix:


,cat,dog,mat,on,sat,the
"""the cat sat on the mat""",1,0,1,1,1,2
"""the dog sat on the cat""",1,1,0,1,1,2


In [34]:
# Using sklearn's CountVectorizer (the professional way)
from sklearn.feature_extraction.text import CountVectorizer

# Create vectorizer
count_vectorizer = CountVectorizer()

# Sample documents
documents = [
    "the cat sat on the mat",
    "the dog sat on the cat",
    "the cat and dog are friends"
]

# Fit and transform
bow_matrix = count_vectorizer.fit_transform(documents)

# Get feature names (vocabulary)
vocab = count_vectorizer.get_feature_names_out()
print("Vocabulary:", vocab)

Vocabulary: ['and' 'are' 'cat' 'dog' 'friends' 'mat' 'on' 'sat' 'the']


In [35]:
bow_matrix.toarray()

array([[0, 0, 1, 0, 0, 1, 1, 1, 2],
       [0, 0, 1, 1, 0, 0, 1, 1, 2],
       [1, 1, 1, 1, 1, 0, 0, 0, 1]])

In [36]:
# View as DataFrame
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vocab,
    index=[f"Doc {i+1}" for i in range(len(documents))]
)
bow_df

,and,are,cat,dog,friends,mat,on,sat,the
Doc 1,0,0,1,0,0,1,1,1,2
Doc 2,0,0,1,1,0,0,1,1,2
Doc 3,1,1,1,1,1,0,0,0,1


In [37]:
# ⚠️ BOW FAILURE MODE: Order is lost!
print("\n🚨 BoW PROBLEM: Word order is completely ignored!")
print()

sentences = [
    "not good",
    "good not"
]

vec = CountVectorizer()
matrix = vec.fit_transform(sentences)

print(f"Vocabulary: {vec.get_feature_names_out()}")
print(f"'not good' → {matrix.toarray()[0]}")
print(f"'good not' → {matrix.toarray()[1]}")
print("\n❌ Same vector! BoW can't distinguish these!")


🚨 BoW PROBLEM: Word order is completely ignored!

Vocabulary: ['good' 'not']
'not good' → [1 1]
'good not' → [1 1]

❌ Same vector! BoW can't distinguish these!


In [38]:
# More failure examples
failure_examples = [
    ("I love to hate it", "I hate to love it"),
    ("dog bites man", "man bites dog"),
    ("this is good", "is this good"),
]

print("BoW treats these pairs as IDENTICAL:\n")
for s1, s2 in failure_examples:
    vec = CountVectorizer()
    matrix = vec.fit_transform([s1, s2])
    identical = np.array_equal(matrix.toarray()[0], matrix.toarray()[1])
    print(f"  '{s1}'")
    print(f"  '{s2}'")
    print(f"  → Same vector: {identical}")
    print()

BoW treats these pairs as IDENTICAL:

  'I love to hate it'
  'I hate to love it'
  → Same vector: True

  'dog bites man'
  'man bites dog'
  → Same vector: True

  'this is good'
  'is this good'
  → Same vector: True



In [39]:
# Apply BoW to our IMDB data
from sklearn.feature_extraction.text import CountVectorizer

# Clean the text first
def clean_text(text):
    # Remove HTML
    text = re.sub(r'<[^>]+>', ' ', text)
    # Lowercase
    text = text.lower()
    return text

df['clean_text'] = df['review'].apply(clean_text)

# Create BoW with max 1000 features (for speed)
vectorizer = CountVectorizer(max_features=1000, stop_words='english')
X_bow = vectorizer.fit_transform(df['clean_text'])

print(f"BoW matrix shape: {X_bow.shape}")
print(f"  → {X_bow.shape[0]} documents")
print(f"  → {X_bow.shape[1]} unique words (features)")

BoW matrix shape: (50000, 1000)
  → 50000 documents
  → 1000 unique words (features)


In [40]:
# Look at the most common words
word_counts = X_bow.sum(axis=0).A1  # Sum across all documents
word_freq = list(zip(vectorizer.get_feature_names_out(), word_counts))
word_freq.sort(key=lambda x: x[1], reverse=True)

print("Top 20 most frequent words in corpus:")
for word, count in word_freq[:20]:
    print(f"  {word}: {int(count)}")

Top 20 most frequent words in corpus:
  movie: 87970
  film: 79705
  like: 40172
  just: 35183
  good: 29753
  time: 25109
  story: 23119
  really: 23094
  bad: 18473
  people: 18188
  great: 18144
  don: 17623
  make: 15898
  way: 15645
  movies: 15309
  characters: 14456
  think: 14337
  watch: 13946
  character: 13905
  films: 13755


In [41]:
# Compare positive vs negative reviews
positive_mask = df['sentiment'] == 'positive'
negative_mask = df['sentiment'] == 'negative'

# Convert pandas Series to numpy arrays for sparse matrix indexing
# This prevents the 'Series' object has no attribute 'nonzero' error
positive_counts = X_bow[positive_mask.values].sum(axis=0).A1
negative_counts = X_bow[negative_mask.values].sum(axis=0).A1

# Create comparison DataFrame
comparison = pd.DataFrame({
    'word': vectorizer.get_feature_names_out(),
    'positive_count': positive_counts,
    'negative_count': negative_counts,
})

# Calculate ratio (positive / negative)
comparison['ratio'] = (comparison['positive_count'] + 1) / (comparison['negative_count'] + 1)

print("\n🟢 Words more common in POSITIVE reviews:")
print(comparison.nlargest(10, 'ratio')[['word', 'positive_count', 'negative_count', 'ratio']])

print("\n🔴 Words more common in NEGATIVE reviews:")
print(comparison.nsmallest(10, 'ratio')[['word', 'positive_count', 'negative_count', 'ratio']])



🟢 Words more common in POSITIVE reviews:
          word  positive_count  negative_count     ratio
847     superb            1116             184  6.037838
973  wonderful            2668             551  4.835145
279  excellent            3359             745  4.504021
303  fantastic            1228             291  4.208904
32     amazing            2004             516  3.878143
665   powerful             960             275  3.481884
98   brilliant            1874             540  3.465804
460    journey             702             205  3.412621
629    perfect            2434             720  3.377254
308   favorite            1843             548  3.358834

🔴 Words more common in NEGATIVE reviews:
          word  positive_count  negative_count     ratio
949      waste             178            2611  0.068530
982      worst             446            4888  0.091430
59       awful             304            3143  0.097010
655     poorly             134            1258  0.107228
650 

## Tf-idf

In [42]:
# TF-IDF = Term Frequency × Inverse Document Frequency
import math

# Simple demonstration
documents = [
    "the movie was great and fun",
    "the movie was terrible",
    "the film was excellent and great"
]

# Total documents
N = len(documents)

# Count how many documents contain each word
word_doc_count = Counter()
for doc in documents:
    unique_words = set(doc.lower().split())
    for word in unique_words:
        word_doc_count[word] += 1

print("Document frequency (how many docs contain each word):")
for word, count in sorted(word_doc_count.items()):
    print(f"  '{word}': {count}/{N} documents")

Document frequency (how many docs contain each word):
  'and': 2/3 documents
  'excellent': 1/3 documents
  'film': 1/3 documents
  'fun': 1/3 documents
  'great': 2/3 documents
  'movie': 2/3 documents
  'terrible': 1/3 documents
  'the': 3/3 documents
  'was': 3/3 documents


In [43]:
# Calculate IDF manually
print("\nIDF calculation: log(N / doc_frequency)")
print(f"N = {N} (total documents)\n")

for word in ['the', 'movie', 'was', 'great', 'excellent', 'terrible']:
    df_word = word_doc_count.get(word, 0)
    if df_word > 0:
        idf = math.log(N / df_word) + 1  # +1 is smoothing
        print(f"  '{word}': appears in {df_word} docs → IDF = log({N}/{df_word})+1 = {idf:.2f}")
    else:
        print(f"  '{word}': not found")

print("\n💡 Key insight: Words in ALL docs get LOW IDF!")
print("   Words in FEW docs get HIGH IDF!")


IDF calculation: log(N / doc_frequency)
N = 3 (total documents)

  'the': appears in 3 docs → IDF = log(3/3)+1 = 1.00
  'movie': appears in 2 docs → IDF = log(3/2)+1 = 1.41
  'was': appears in 3 docs → IDF = log(3/3)+1 = 1.00
  'great': appears in 2 docs → IDF = log(3/2)+1 = 1.41
  'excellent': appears in 1 docs → IDF = log(3/1)+1 = 2.10
  'terrible': appears in 1 docs → IDF = log(3/1)+1 = 2.10

💡 Key insight: Words in ALL docs get LOW IDF!
   Words in FEW docs get HIGH IDF!


In [44]:
# Using sklearn's TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer()

documents = [
    "the movie was great and fun",
    "the movie was terrible",
    "the film was excellent and great"
]

# Transform
X_tfidf = tfidf_vectorizer.fit_transform(documents)

# View as DataFrame
tfidf_df = pd.DataFrame(
    X_tfidf.toarray().round(2),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(documents))]
)

print("TF-IDF Matrix (values are weights, not raw counts):")
tfidf_df

TF-IDF Matrix (values are weights, not raw counts):


,and,excellent,film,fun,great,movie,terrible,the,was
Doc 1,0.41,0.00,0.00,0.54,0.41,0.41,0.00,0.32,0.32
Doc 2,0.00,0.00,0.00,0.00,0.00,0.50,0.66,0.39,0.39
Doc 3,0.39,0.51,0.51,0.00,0.39,0.00,0.00,0.30,0.30


In [45]:
# Apply TF-IDF to IMDB dataset
print("Applying TF-IDF to IMDB dataset...\n")

# TF-IDF with unigrams + bigrams
tfidf_vec = TfidfVectorizer(
    max_features=2000,
    stop_words='english',
    ngram_range=(1, 2)  # Unigrams and bigrams!
)

X_tfidf = tfidf_vec.fit_transform(df['clean_text'])

print(f"TF-IDF matrix shape: {X_tfidf.shape}")
print(f"  → {X_tfidf.shape[0]} documents")
print(f"  → {X_tfidf.shape[1]} features (unigrams + bigrams)")

Applying TF-IDF to IMDB dataset...

TF-IDF matrix shape: (50000, 2000)
  → 50000 documents
  → 2000 features (unigrams + bigrams)


In [46]:
# Find words with highest IDF (most distinctive)
idf_scores = tfidf_vec.idf_
feature_names = tfidf_vec.get_feature_names_out()

idf_df = pd.DataFrame({
    'feature': feature_names,
    'idf': idf_scores
}).sort_values('idf', ascending=False)

print("Features with HIGHEST IDF (most distinctive/rare):")
print(idf_df.head(15))

print("\nFeatures with LOWEST IDF (most common):")
print(idf_df.tail(10))

Features with HIGHEST IDF (most distinctive/rare):
       feature       idf
837     hitler  6.351738
959     keaton  6.326737
155     batman  6.225087
64       alice  6.217679
1866  vampires  6.210326
713         fu  6.210326
63        alex  6.203027
1811      trek  6.106065
690   football  6.096213
399        dan  6.092951
1921     wayne  6.092951
1082     lynch  6.092951
82         ann  6.086457
76        andy  6.086457
936      jesus  6.073595

Features with LOWEST IDF (most common):
     feature       idf
771    great  2.380173
480      don  2.349173
1681   story  2.212697
1432  really  2.200798
1784    time  2.059431
756     good  1.967341
953     just  1.867902
1027    like  1.769357
647     film  1.588339
1176   movie  1.492122


In [48]:
# Step 1: Imports and Train/Test Split
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Encode labels: positive -> 1, negative -> 0
y = (df['sentiment'] == 'positive').astype(int)

# Use the cleaned text we built earlier
X_text = df['clean_text']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train_text)}  |  Test size: {len(X_test_text)}")
print(f"Train positive ratio: {y_train.mean():.2f}")
print(f"Test  positive ratio: {y_test.mean():.2f}")

Train size: 40000  |  Test size: 10000
Train positive ratio: 0.50
Test  positive ratio: 0.50


In [51]:
from sklearn.metrics import accuracy_score
cbow = CountVectorizer(max_features=5000, stop_words='english')

X_train_bow = cbow.fit_transform(X_train_text)
X_test_bow = cbow.transform(X_test_text)

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)


# train logistic regression on BoW
lr_bow = LogisticRegression()
lr_bow.fit(X_train_bow, y_train)
y_pred_bow = lr_bow.predict(X_test_bow)
print("Logistic Regression with BoW:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_bow):.2f}")

#print(classification_report(y_test, y_pred_bow))

# train logistic regression on TF-IDF
lr_tfidf = LogisticRegression()
lr_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = lr_tfidf.predict(X_test_tfidf)
print("Logistic Regression with TF-IDF:")
#print(classification_report(y_test, y_pred_tfidf))
print(f"Accuracy: {accuracy_score(y_test, y_pred_tfidf):.2f}")

Logistic Regression with BoW:
Accuracy: 0.87
Logistic Regression with TF-IDF:
Accuracy: 0.89


/Users/shivam13juna/Documents/virtual_envs/dev3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [52]:
lr_bow.predict(X_test_bow)[:10]

array([0, 0, 1, 0, 0, 0, 0, 1, 0, 0])

# Keras with BOW

In [53]:
# A tiny toy corpus — just two sentences so the matrix is easy to read.
sentences = [
    "the cat sat on the mat",
    "dogs bark at night",
    #"the sun rises in the east",
    #"birds fly high in the sky",
    #"children play games after school"
]

# Step 1: tokenize
# 'tokenize' = split a sentence into individual words (tokens).
# We also lowercase everything so 'The' and 'the' are treated as the same word.
tokenized = [s.lower().split() for s in sentences]

# Step 2: build the vocabulary (the unique set of words across all sentences)
# We sort it so the order is deterministic — same vocab order every time we run.
vocab = sorted(set(word for sent in tokenized for word in sent))

# Step 3: assign each word a unique integer id (and keep the reverse mapping too)
# Neural nets can't process strings — they need numbers. word_to_id does that mapping.
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for w, i in word_to_id.items()}
vocab_size = len(vocab)

print("Vocabulary:", word_to_id)

Vocabulary: {'at': 0, 'bark': 1, 'cat': 2, 'dogs': 3, 'mat': 4, 'night': 5, 'on': 6, 'sat': 7, 'the': 8}


In [54]:
# numpy: numerical arrays. Keras/TensorFlow expects inputs as numpy arrays (or tensors).
import numpy as np
# tensorflow: the deep-learning framework. Keras lives inside tf as `tf.keras`.
import tensorflow as tf
# 'layers' is a shortcut to all the layer classes (Dense, Embedding, etc.).
from tensorflow.keras import layers
# `chain` flattens a list of lists into a single iterable. Handy for vocab building.
from itertools import chain

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.18.1


In [55]:
# A slightly bigger toy corpus so the model has a few patterns to learn from.
# Real Word2Vec is trained on BILLIONS of words; this is just for demonstration.
sentences = [
    "the cat sat on the mat in the evening quietly",
    "dogs bark loudly when they see strangers near the gate",
    "birds fly in the sky and sing songs in the morning",
    "the sun rises in the east and sets in the west",
    "children play games in the park after school hours",
    "the teacher teaches maths and science in the classroom",
    "books are kept neatly on the shelf by the librarian",
    "the gardener waters the plants in the garden daily",
    "parents drop their kids at school every morning",
    "students read and write silently during the study hour",
]

# Tokenize: split each sentence into lowercase words.
tokenized = [s.lower().split() for s in sentences]

# Build the vocabulary by flattening all sentences into one big list of words,
# then taking the unique set, then sorting (for reproducibility).
vocab = sorted(set(chain(*tokenized)))

# Word <-> integer-id mappings (neural networks need numeric inputs).
word_to_ix = {w: i for i, w in enumerate(vocab)}
ix_to_word = {i: w for w, i in word_to_ix.items()}
vocab_size = len(vocab)

print("Vocabulary size:", vocab_size)

Vocabulary size: 66


In [58]:
# For CBOW, every training example is: (context words, target word).
# We slide a 'window' across each sentence and at every position we record:
#   - the 2 words BEFORE + 2 words AFTER the center word -> context
#   - the center word itself -> target

window_size = 2  # 2 words on each side -> 4 context words per example

def generate_cbow_pairs(tokenized_sentences, window_size=2):
    pairs = []
    for sentence in tokenized_sentences:
        # We start at index `window_size` (so we have enough words on the LEFT)
        # and stop `window_size` words before the end (enough words on the RIGHT).
        for i in range(window_size, len(sentence) - window_size):
            # Build the context: window_size words to the left + window_size words to the right.
            # The reversed range on the left keeps left-context in original order.
            context = [sentence[i - j] for j in range(window_size, 0, -1)] + \
                      [sentence[i + j] for j in range(1, window_size + 1)]
            target = sentence[i]
            pairs.append((context, target))
    return pairs

cbow_data = generate_cbow_pairs(tokenized, window_size)

# Print the first few pairs to sanity-check what we built.
print("Sample CBOW pairs (context -> target):")
for context, target in cbow_data:
    print(context, "->", target)

Sample CBOW pairs (context -> target):
['the', 'cat', 'on', 'the'] -> sat
['cat', 'sat', 'the', 'mat'] -> on
['sat', 'on', 'mat', 'in'] -> the
['on', 'the', 'in', 'the'] -> mat
['the', 'mat', 'the', 'evening'] -> in
['mat', 'in', 'evening', 'quietly'] -> the
['dogs', 'bark', 'when', 'they'] -> loudly
['bark', 'loudly', 'they', 'see'] -> when
['loudly', 'when', 'see', 'strangers'] -> they
['when', 'they', 'strangers', 'near'] -> see
['they', 'see', 'near', 'the'] -> strangers
['see', 'strangers', 'the', 'gate'] -> near
['birds', 'fly', 'the', 'sky'] -> in
['fly', 'in', 'sky', 'and'] -> the
['in', 'the', 'and', 'sing'] -> sky
['the', 'sky', 'sing', 'songs'] -> and
['sky', 'and', 'songs', 'in'] -> sing
['and', 'sing', 'in', 'the'] -> songs
['sing', 'songs', 'the', 'morning'] -> in
['the', 'sun', 'in', 'the'] -> rises
['sun', 'rises', 'the', 'east'] -> in
['rises', 'in', 'east', 'and'] -> the
['in', 'the', 'and', 'sets'] -> east
['the', 'east', 'sets', 'in'] -> and
['east', 'and', 'in', 't

In [60]:
# The model can't take strings as input, so we convert words -> integer ids.
# X[i] is a list of 4 ids (the context words for example i).
# y[i] is a single id (the target word for example i).

# dtype=np.int32 is required because Embedding layers expect integer indices.
X = np.array([[word_to_ix[w] for w in ctx] for ctx, _ in cbow_data], dtype=np.int32)
y = np.array([word_to_ix[tgt] for _, tgt in cbow_data], dtype=np.int32)

print("X shape:", X.shape, "  (num_examples, 2*window_size)")
print("y shape:", y.shape)
print("First example -> X:", X[0], " y:", y[0])

X shape: (56, 4)   (num_examples, 2*window_size)
y shape: (56,)
First example -> X: [59  8 35 59]  y: 43


In [61]:
# 'embedding_dim' = size of each word's vector representation.
# Bigger = more expressive, but more parameters to learn. 50 is fine for a toy corpus.
embedding_dim = 50

# Sequential = a simple stack of layers, one after another.
cbow_model = tf.keras.Sequential([
    # Input: each example is a list of 2*window_size = 4 word ids.
    layers.Input(shape=(2 * window_size,)),
	
    # Embedding layer: a lookup table of shape (vocab_size, embedding_dim).
    # It converts each word id into a dense vector of size embedding_dim.
    # Output shape: (batch_size, 4, embedding_dim).
    # We give it a name so we can grab the learned weights later.
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, name="embedding"),
	
    # GlobalAveragePooling1D: averages the 4 context vectors into ONE vector.
    # Output shape: (batch_size, embedding_dim).
    # This is the 'bag of words' part — the order of context words doesn't matter,
    # we just average them all together.
    layers.GlobalAveragePooling1D(),
	
    # Final dense layer: project the averaged context vector to a probability
    # over the entire vocabulary. softmax makes the outputs sum to 1.
    # Output shape: (batch_size, vocab_size).
    layers.Dense(vocab_size, activation="softmax"),
])

#   - accuracy: helpful sanity-check metric (prediction == target?).
cbow_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# summary() prints the layer-by-layer architecture and parameter count.
cbow_model.summary()

2026-05-09 21:54:03.674109: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Max
2026-05-09 21:54:03.674156: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-05-09 21:54:03.674161: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 53.76 GB
I0000 00:00:1778343843.674561 87281191 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1778343843.674755 87281191 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 4, 50)          │         3,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 50)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 66)             │         3,366 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,666 (26.04 KB)

 Trainable params: 6,666 (26.04 KB)

 Non-trainable params: 0 (0.00 B)

In [62]:
history = cbow_model.fit(
    X, y,
    epochs=100,
    batch_size=8,
    shuffle=True,
    verbose=0,
)

# `history.history` is a dict: { 'loss': [...], 'accuracy': [...] } — one value per epoch.
# We print every 10th epoch to mimic the original PyTorch notebook's output style.
for epoch in range(0, 100, 10):
    print(f"Epoch {epoch} Loss: {history.history['loss'][epoch]:.4f}")

2026-05-09 21:54:14.642154: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


Epoch 0 Loss: 4.1681
Epoch 10 Loss: 1.2007
Epoch 20 Loss: 0.2687
Epoch 30 Loss: 0.1052
Epoch 40 Loss: 0.0639
Epoch 50 Loss: 0.0487
Epoch 60 Loss: 0.0450
Epoch 70 Loss: 0.0384
Epoch 80 Loss: 0.0330
Epoch 90 Loss: 0.0337


In [63]:
# After training, the Embedding layer's weight matrix IS the word2vec embeddings.
# get_weights() returns a list; for an Embedding layer the only weight is the
# lookup table of shape (vocab_size, embedding_dim).
embedding_matrix = cbow_model.get_layer("embedding").get_weights()[0]

# To get the vector for a specific word, look up its row in the matrix.
word = "school"
vec = embedding_matrix[word_to_ix[word]]
print(f"Embedding vector for '{word}' (dim={vec.shape[0]}):\n", vec)

Embedding vector for 'school' (dim=50):
 [ 0.5755141  -0.02665148 -1.1413916  -0.5707307  -0.3112572  -0.9411609
  0.29769155 -0.10419335 -0.58727866  0.6345038  -1.1132666   0.74357927
 -0.7805894   1.137058    1.1917202  -1.0625092   0.33328766 -0.532825
  0.09232683 -0.76283354 -0.2868436   0.09944189  0.04955879  0.1460712
  0.30151576  0.9514013  -0.6812933  -0.95329     0.29516417 -1.0259455
  0.755949   -0.91963995  0.14062424  0.5724566  -0.6219696  -0.01688432
 -0.19110148 -1.1182821   0.9706839  -0.8856808  -0.20989303  0.10834508
 -0.9389698  -0.75969625  0.50475615 -0.5334391   1.0958816   0.39602014
  0.36145437 -0.37483466]


In [64]:
context_words = ["the", "cat", "on", "the"]

# model.predict expects a BATCH of examples, even if there's just one.
# So we wrap our 4-id list inside another list -> shape (1, 4).
context_ids = np.array([[word_to_ix[w] for w in context_words]])

# predict() returns probabilities over the whole vocabulary, shape (1, vocab_size).
# [0] picks the first (and only) example in the batch.
probs = cbow_model.predict(context_ids, verbose=0)[0]

# argmax picks the index with the highest probability — the model's top guess.
predicted_id = probs.argmax()
print(f"Context {context_words} -> predicted target: '{ix_to_word[predicted_id]}'")

Context ['the', 'cat', 'on', 'the'] -> predicted target: 'sat'


In [67]:
np.round(probs, 3)

array([0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.   , 0.002, 0.   , 0.   , 0.   , 0.   , 0.001,
       0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.993, 0.   ,
       0.   , 0.   , 0.   , 0.002, 0.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.   ], dtype=float32)

In [68]:
predicted_id

np.int64(43)

In [69]:
ix_to_word[predicted_id]

'sat'

1. What the concepts are? doable in class
2. Technical fluency: convert ideas in your head to code. not doable in class, this requires practice and experience.

## Skip-gram

In [71]:

# Build (center, context) pairs.
# For each word in each sentence, pair it with EVERY word inside its window.

def generate_skipgram_pairs(tokenized_sentences, window_size=2):
    pairs = []
    for tokens in tokenized_sentences:
        sent_len = len(tokens)

        # Treat every word in the sentence as a potential center word.
        for idx, center in enumerate(tokens):
            # Compute the window boundaries, clamped to the sentence edges.
            # max(.., 0) prevents going off the LEFT side (negative index).
            # min(.., sent_len) prevents going off the RIGHT side.
            start = max(idx - window_size, 0)
            end = min(idx + window_size + 1, sent_len)

            # LEFT context: words from `start` up to (but not including) the center.
            pairs.extend((center, ctx) for ctx in tokens[start:idx])
            # RIGHT context: words from just after the center up to `end`.
            pairs.extend((center, ctx) for ctx in tokens[idx + 1:end])
    return pairs

skipgram_data = generate_skipgram_pairs(tokenized, window_size)

print("Sample Skip-Gram pairs (center -> context):")
for center, ctx in skipgram_data:
    print(center, "->", ctx)

Sample Skip-Gram pairs (center -> context):
the -> cat
the -> sat
cat -> the
cat -> sat
cat -> on
sat -> the
sat -> cat
sat -> on
sat -> the
on -> cat
on -> sat
on -> the
on -> mat
the -> sat
the -> on
the -> mat
the -> in
mat -> on
mat -> the
mat -> in
mat -> the
in -> the
in -> mat
in -> the
in -> evening
the -> mat
the -> in
the -> evening
the -> quietly
evening -> in
evening -> the
evening -> quietly
quietly -> the
quietly -> evening
dogs -> bark
dogs -> loudly
bark -> dogs
bark -> loudly
bark -> when
loudly -> dogs
loudly -> bark
loudly -> when
loudly -> they
when -> bark
when -> loudly
when -> they
when -> see
they -> loudly
they -> when
they -> see
they -> strangers
see -> when
see -> they
see -> strangers
see -> near
strangers -> they
strangers -> see
strangers -> near
strangers -> the
near -> see
near -> strangers
near -> the
near -> gate
the -> strangers
the -> near
the -> gate
gate -> near
gate -> the
birds -> fly
birds -> in
fly -> birds
fly -> in
fly -> the
in -> birds
in 

the cat sat on the mat